# VQE on the 2D Transverse-Field Ising Model via Pauli Propagation

This tutorial demonstrates a full **Variational Quantum Eigensolver (VQE)** workflow using `pprop` on a non-trivial many-body problem: the 2D transverse-field Ising model on a $5 \times 5$ lattice with open boundary conditions.

VQE minimises the energy expectation value $\langle H \rangle$ over a parametrized ansatz $U(\boldsymbol{\theta})$:

$$E(\boldsymbol{\theta}) = \langle 0 | U^\dagger(\boldsymbol{\theta})\, H\, U(\boldsymbol{\theta}) | 0 \rangle$$

With `pprop`, $E(\boldsymbol{\theta})$ is computed as a closed-form trigonometric polynomial via Pauli propagation, no statevector simulation required. This makes gradient evaluation cheap even for 25 qubits.

## Step 0: Imports

In [ ]:
import pennylane as qml
from pprop import Propagator
import numpy as np

## Step 1: Define the Hamiltonian

The **2D transverse-field Ising model** on an $L \times L$ lattice with open boundary conditions is:

$$H(J, h) = -\frac{1}{N}\left(J\sum_{\langle i,j \rangle} Z_i Z_j + h\sum_i X_i\right)$$

where $N = L^2$ is the total number of sites, $\langle i,j \rangle$ denotes nearest-neighbour pairs (right and down neighbours only, to avoid double-counting), $J$ is the ferromagnetic coupling strength, and $h$ is the transverse field strength. The $1/N$ normalisation keeps the energy per site of order 1 regardless of system size.

We use $J = h = 1$. In 2D the critical point lies at $h/J \approx 3.04$, so this choice places the system well inside the **ferromagnetic phase** ($h \ll h_c$), where the ground state has strong $ZZ$ correlations and the transverse field acts as a perturbation.

In [ ]:
side : int = 5
J : float = 1
h : float = 1

num_qubits : int = side * side

In [ ]:
def hamiltonian(side: int, J: float, h: float) -> qml.Hamiltonian:
    coeffs = []
    obs = []

    # Nearest-neighbor ZZ interactions
    for x in range(side):
        for y in range(side):
            i = x * side + y

            # Right neighbor
            if y < side - 1:
                j = x * side + (y + 1)
                coeffs.append(-J / num_qubits)
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

            # Down neighbor
            if x < side - 1:
                j = (x + 1) * side + y
                coeffs.append(-J / num_qubits)
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    # Transverse-field X terms
    for i in range(num_qubits):
        coeffs.append(-h / num_qubits)
        obs.append(qml.PauliX(i))

    return qml.Hamiltonian(coeffs, obs)

The Hamiltonian in full:

$$H(J, h) = -\frac{1}{N}\left(J\sum_{\langle i,j \rangle} Z_i Z_j + h\sum_i X_i\right)$$

For our $5 \times 5$ lattice ($N = 25$), this yields $40$ nearest-neighbour $ZZ$ terms (20 horizontal + 20 vertical) and $25$ single-site $X$ terms, **65 Pauli words** in total.

The lattice connectivity for the $5 \times 5$ open boundary condition model. Each node is a qubit; edges represent $ZZ$ couplings:

<img src="../assets/ising2d.svg" width="600">

## Step 2: Define the Ansatz

We use a hardware-efficient ansatz designed to respect the 2D lattice geometry. The circuit structure is:

1. **Initial rotation layer:** RY + RX on every qubit: 50 parameters
2. **Horizontal entanglers:** two rounds of CNOT gates along rows (even columns, then odd columns), covering all horizontal nearest-neighbour pairs
3. **RX layer:** single-qubit RX on every qubit: 25 parameters
4. **Vertical entanglers:** two rounds of CNOT gates along columns (even rows, then odd rows), covering all vertical nearest-neighbour pairs
5. **RX layer:** single-qubit RX on every qubit: 25 parameters
6. **Final RY layer:** single-qubit RY on every qubit: 25 parameters

Total: **125 trainable parameters** across 25 qubits. The two-round CNOT strategy (even-then-odd) ensures every nearest-neighbour pair is entangled.

In [ ]:
def circuit(params):
    index = 0

    # Initial RY and RX
    for q in range(num_qubits):
        qml.RY(params[index], wires=q)
        index += 1
        qml.RX(params[index], wires=q)
        index += 1

    # Horizontal entanglers
    for d in range(2):
        y_start = 0 if d % 2 == 0 else 1
        for x in range(side):
            for y in range(y_start, side - 1, 2):
                i = x * side + y
                j = x * side + (y + 1)
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(num_qubits):
        qml.RX(params[index], wires=q)
        index += 1

    # Vertical entanglers
    for d in range(2):
        x_start = 0 if d % 2 == 0 else 1
        for y in range(side):
            for x in range(x_start, side - 1, 2):
                i = x * side + y
                j = (x + 1) * side + y
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(num_qubits):
        qml.RX(params[index], wires=q)
        index += 1

    # Final RY
    for q in range(num_qubits):
        qml.RY(params[index], wires=q)
        index += 1
        
    return qml.expval(hamiltonian(side, J, h))

## Step 3: Create the Propagator

`Propagator` records the ansatz onto a quantum tape and prepares it for Heisenberg-picture propagation. No truncation is used here, the exact pruning strategies enabled in the next step are enough to keep propagation fast without discarding any terms.

In [ ]:
prop = Propagator(circuit)

In [ ]:
prop

We can inspect the full circuit structure using `.show()`. The 25-qubit circuit is displayed as a series of horizontal slices:

In [ ]:
prop.show()

## Step 4: Propagate

`.propagate()` evolves the full Hamiltonian backwards through the circuit. Since the Hamiltonian is a sum of 65 Pauli words, each is propagated independently and the results are combined, entirely inside the Rust extension `pprop_rs`, which returns each observable's final propagated expression (`prop.exprs`) rather than any intermediate state, so this cell itself has no output to inspect.

`use_dead_qubit_pruner=True, use_xy_weight_pruner=True` enable the two exact pruning strategies (see the pruning/truncation tutorial notebook): they discard Pauli words that provably cannot contribute to the result, so propagation stays fast with no approximation involved.

In [ ]:
prop.propagate(use_dead_qubit_pruner=True, use_xy_weight_pruner=True)

## Step 5: Run VQE

With the propagator ready, VQE reduces to minimising a fast scalar function. We use the `adam` optimiser from `pprop.optimization`, which:
1. Calls `prop.eval_and_grad(params)` to get $E(\boldsymbol{\theta})$ and $\nabla_{\boldsymbol{\theta}} E$ analytically
2. Passes the gradient to an Adam update step via `optax`

The loss `L = lambda f: f[0]` simply selects the first (and only) observable, the Hamiltonian expectation value, as the quantity to minimise.

We run for **2000 steps** with learning rate $10^{-3}$ from a random initial point.

In [ ]:
from pprop.optimization import adam

result = adam(
    L=lambda f: f[0],                  # minimise the energy expectation value
    propagator=prop,
    params_init=np.random.rand(prop.num_params),
    lr=1e-3,
    num_steps=2000
)

The optimisation converges close to the DMRG reference plotted below. Since propagation here is exact (no truncation), the remaining gap to the true ground state comes only from the ansatz's limited expressivity and the optimisation itself.

The result dict contains the final parameters, the converged loss, and the full loss history for plotting:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(result['loss_history'])
ax.axhline(-1.76649063, color='r', linestyle='--', label='DMRG ($E = -1.7665$)')
ax.set_xlabel('Step')
ax.set_ylabel('Energy')
ax.set_title('VQE convergence 2D TFIM $5 \\times 5$')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final energy : {result['fun']:.6f}")
print("DMRG energy  : -1.76649063")
print(f"Gap          : {result['fun'] - (-1.76649063):.6f}")